# Noise ablation — which of our two noise sources is doing the work?

**Direction:** `research/directions/noise-ablation.md` · `[in-frame]` · sub-Q 1, 2, 3. **Model:** GRU only, `H = 256`.

**The gap.** Every dataset in this repo carries **two independent noise sources**, and no result has ever separated
them:

| source | config field | standard value | what it corrupts |
|---|---|---|---|
| **observation noise** | `obs_noise_std` | 0.2 | the 1D intensity scan — **sensing**. The world is exact; the model's view of it is not. |
| **position noise** | `position_noise_std` | 0.04 | the discs' positions each step (Gaussian diffusion on top of constant-velocity drift) — **the world itself**. The future is genuinely uncertain however well it is sensed. |

They are conceptually opposite: sensing noise should push the latent toward *averaging and filtering*, while world noise
caps how much long-horizon structure is worth representing at all. We have been reading every finding off the both-on
cell of a 2×2 we never filled in. This notebook fills it in.

**Provenance.** Three new datasets were generated matched to `4_fixed_refl_inview` in every respect except the two noise
flags; four models trained by `scripts/train_gru.py` with an identical recipe; the metric suite computed once by
`scripts/eval_controls.py` and only loaded and plotted here.

## Definitions

### Runs and datasets (copied from `CONTROL_RUNS.md`, per the repo's run-registry rule)

Every run: `hidden_size = 256`, 400 epochs, batch 256, AdamW lr 1e-3, weight decay 1e-4, seed 0, 1 GRU layer.
Every dataset: 2 objects, 40 frames, 128 rays, open boundary, fixed reflectivities, always-in-frustum,
90k/10k/10k/10k splits, base seed 0, edit frame 20. **The only variables are the two noise flags.**

| code | descriptive label (used in every figure) | dataset | observation noise | position noise |
|---|---|---|---|---|
| `N_obs0_pos0` | **no noise** (obs 0.0, pos 0.00) | `9_obsnoise0_posnoise0` | 0.0 | 0.00 |
| `N_obs0_pos004` | **world noise only** (obs 0.0, pos 0.04) | `10_obsnoise0_posnoise004` | 0.0 | 0.04 |
| `N_obs02_pos0` | **sensing noise only** (obs 0.2, pos 0.00) | `11_obsnoise02_posnoise0` | 0.2 | 0.00 |
| `H256` | **both noises** (obs 0.2, pos 0.04) — *the repo standard* | `4_fixed_refl_inview` | 0.2 | 0.04 |

Cells are ordered **no noise → world noise only → sensing noise only → both** in every figure and table, with one fixed
colour per cell, so a cell can be tracked across panels without re-reading the legend.

> ### ⚠ Absolute RMSE is NOT comparable across these four cells
> A model trained and evaluated on noise-free observations has an observation noise floor of ~0, so its raw RMSE is
> lower for bookkeeping reasons, not because it is a better world model. **Every predictive number is read against that
> model's own baselines** — which is why Fig 1 plots each cell's own noise floor and adds a panel normalised by each
> cell's own copy-previous-frame baseline. The same mistake cost the endogenous thread a set of cross-citable numbers
> (see `../actions/ENDOGENOUS_RUNS.md`).
>
> Each model is also evaluated on **its own** dataset's edits split — the in-distribution choice. The four cells
> therefore contain different scenes, so the waterfall (Fig 5) gives **each cell its own GT column** and the scalar
> metrics are matched statistically (64 random edits from matched generators), not sample-by-sample.

### Metrics (formulas verbatim from `../METRICS_AND_EDITORS.md`)

| name | formula | units | better |
|---|---|---|---|
| next-step RMSE | `RMSE(pred_t, clean_obs[t+1])`, teacher-forced | obs intensity [0,1] | ↓ |
| free-run RMSE @ step s | warm up on `obs[0..9]`, then free-run; `RMSE(roll_s, clean_obs[10+s])` | obs intensity | ↓ |
| copy-previous-frame / noise floor / random frame | **that cell's own** dataset baselines, `pim/eval/baselines.py` | obs intensity | reference lines |
| position / velocity R² | `1 − ‖Y − probe(h)‖²/‖Y − Ȳ‖²`, held-out 30%, frames where both objects are visible | — | ↑ |
| fiber residual | `‖h − g(pos,vel)‖ / ‖h‖`, `g` linear or MLP, held-out 30% | fraction of ‖h‖ | ↓ (0 = fully canonical) |
| **Edit Index** | `(d_uned − d_edit)/(d_uned + d_edit)`, `d_· = RMSE(edited₀, gt_·)` over the **differing rays**; per sample, then averaged | −1…+1 | ↑ |
| **Target RMSE** | `RMSE(edited₀, gt_edited)` over **target rays** | obs intensity | ↓ |
| **Ghost RMSE** | `RMSE(edited₀, gt_edited)` over **ghost rays** | obs intensity | ↓ |
| **Collateral RMSE** | `RMSE(edited₀, gt_edited)` over **collateral rays** | obs intensity | ↓ |
| **Edit-frame RMSE** | `RMSE(edited₀, gt_edited)` over **all rays** | obs intensity | ↓ |
| **GT-traj RMSE** | `mean_s RMSE(edited_s, clean_obs[ef+s])` over the K-step rollout | obs intensity | ↓ |
| **fidelity ratio** | `GT-traj RMSE(editor) / GT-traj RMSE(unsteered)` | ratio | ↓ (**> 1 = the edit left the rollout FURTHER from the true post-edit world than doing nothing**) |

**The two ground-truth worlds.** Every §4 number is an error against **ground truth**, never against the unsteered
rollout. At the edit frame both worlds are rendered: **`gt_edited`** (= `clean_obs[ef]`, the teleport happened) and
**`gt_unedited`** (the counterfactual where it did not — the edited object continued from its `ef−1` position along its
own velocity, the other object at its true `ef` position).

**Ray zones (per sample, derived from those two renders, so occlusion needs no special-casing).** `target rays` = rays
the edited object occupies in `gt_edited`; `ghost rays` = rays it occupied pre-edit and now vacates; `collateral rays`
= the **other** object's rays (it must not move); `differing rays` = every ray where the two worlds differ — the
support of the Edit Index. **`ef` = 20**, **K = 15** rollout steps, **64** edit samples, **2000** test sequences for
probes.

> **How to read the Edit Index.** **+1** = the output *is* the edited world · **0** = equidistant from both (ambiguous,
> or garbage) · **−1** = the output *is* the unedited world. Unsteered lands near −1 by construction, so the scale is
> anchored on two ground truths rather than on a model-dependent reference. Crucially, an output far from *both* worlds
> — a scrambled or collapsed rollout — scores **≈ 0** rather than a spuriously good value, so the index cannot be gamed
> by destroying the output; the accompanying zone RMSEs then show *how* it was destroyed.
>
> Definitions: `../METRICS_AND_EDITORS.md` §4 · implementation: `scripts/editability_metrics.py` (imported, not
> re-derived). This set **replaces** the retired `reach % of swap` / `collateral % of swap` / `selectivity` /
> `ghost ratio` as of 2026-07-30 — their numbers are **not** comparable to these.

### Editors and references (the standard §4 suite)

**References (never editors):** **GT (sim)** — the simulator's time-evolving clean post-edit observations;
**Unsteered** — free-run from the un-edited warm-up state; **Oracle observation** — teacher-force the model on the true
post-edit observation; the 100% denominator for reach and a *soft* reference.

| editor | mechanism |
|---|---|
| Readout injection | linear pseudoinverse — set the position probe's readout, preserving its null space |
| Global-PCA projection | POCS: alternate inject ↔ project onto the global 99%-variance PCA subspace of visited `h` |
| PCA geodesic | re-project onto a fresh **local** PCA tangent (k=256 neighbours) each step — the canonical structural editor |
| MLP-probe gradient | Adam on `h` through a frozen MLP position probe until it reads the target |
| Decoder gradient (**oracle**) | Adam on `h` to match the true edit-frame observation through the decoder. Not an edit interface — the **bracket**: does a state that renders the target exist and roll out at all? |

> **±1 alignment.** Warm-up teacher-forces `obs[0..ef−1]`, so a rollout's **step 0 decodes sim frame `ef`**
> (`ROLL[:,0] ↔ clean_obs[ef]`). The oracle observation is fed `obs[ef]` and therefore **leads by one frame**.

In [ ]:
# [1] Setup: load the pre-computed metric suite (scripts/eval_controls.py) for all four noise cells.
import os, sys, json
sys.path.insert(0, "../../../..")
import numpy as np, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display, Markdown
from pim.figures.theme import style_ax

OUT = "/tmp/noise_ablation"; os.makedirs(OUT, exist_ok=True)
EVAL = "../../../../runs/controls/eval"

# fixed order: no noise -> world noise only -> sensing noise only -> both
RUNS  = ["N_obs0_pos0", "N_obs0_pos004", "N_obs02_pos0", "H256"]
LABEL = {"N_obs0_pos0":  "no noise\n(obs 0.0, pos 0.00)",
         "N_obs0_pos004":"world noise only\n(obs 0.0, pos 0.04)",
         "N_obs02_pos0": "sensing noise only\n(obs 0.2, pos 0.00)",
         "H256":         "both noises\n(obs 0.2, pos 0.04)"}
SHORT = {k: v.replace("\n", " ") for k, v in LABEL.items()}
COLOR = {"N_obs0_pos0": "#0072B2", "N_obs0_pos004": "#009E73",
         "N_obs02_pos0": "#E69F00", "H256": "#D55E00"}
EDITORS = ["Readout injection", "Global-PCA projection", "PCA geodesic",
           "MLP-probe gradient", "Decoder gradient"]
STRUCTURAL = EDITORS[:4]
ORACLE = "Decoder gradient"

RES = {r: json.load(open(f"{EVAL}/{r}.json")) for r in RUNS}
NPZ = {r: np.load(f"{EVAL}/{r}_rollouts.npz") for r in RUNS}
K   = len(RES["H256"]["editability"]["PCA geodesic"]["step_rmse_to_gt"])
ef  = int(NPZ["H256"]["edit_frame"][0])

print(f"loaded {len(RUNS)} noise cells | K={K} rollout steps | edit frame ef={ef}\n")
hdr = f"{'cell':<34s}{'obs':>6s}{'pos':>7s}{'val_loss':>11s}{'next-step':>11s}{'copy-prev':>11s}{'noise floor':>13s}"
print(hdr); print("-"*len(hdr))
for r in RUNS:
    d = RES[r]; b = d["baselines"]
    print(f"{SHORT[r]:<34s}{d['obs_noise_std']:>6.2f}{d['position_noise_std']:>7.2f}"
          f"{d['val_loss']:>11.5f}{d['nextstep_rmse_vs_clean']:>11.4f}"
          f"{b['identity_rmse']:>11.4f}{b['noise_floor_rmse']:>13.4f}")
print("\nNote how far apart the baselines are -- this is why raw RMSE must never be compared across cells.")

---
## §1 — Predictive quality

Panel (b) shows raw free-run RMSE with **each cell's own** noise floor as a matching dotted line. Panel (c) is the
comparable view: the same curves divided by **each cell's own** copy-previous-frame baseline, so a value below 1 means
"beats the trivial predictor on its own terms" regardless of how noisy that cell is.

In [ ]:
# [2] Fig 1 — predictive quality: (a) validation curves, (b) raw free-run RMSE with per-cell floors, (c) normalised.
curves = {}
for r in RUNS:
    hist = [json.loads(l) for l in open(f"../../../../runs/controls/{r}/metrics.jsonl")]
    curves[r] = ([h["epoch"] for h in hist], [h["train_loss"] for h in hist], [h["val_loss"] for h in hist])

plt.style.use("default")
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.4))
for r in RUNS:
    e, tr, va = curves[r]
    ax[0].plot(e, va, color=COLOR[r], lw=1.5, label=SHORT[r])
    ax[0].plot(e, tr, color=COLOR[r], lw=0.8, ls=":", alpha=0.7)
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("teacher-forced MSE vs next observed frame"); ax[0].set_yscale("log")
ax[0].set_title("(a) training curves\n(solid = validation, dotted = train)", fontsize=10)
ax[0].legend(fontsize=7); ax[0].grid(alpha=0.3); style_ax(ax[0])

s = np.arange(len(RES["H256"]["freerun_rmse_by_step"]))
for r in RUNS:
    fr = np.array(RES[r]["freerun_rmse_by_step"]); b = RES[r]["baselines"]
    ax[1].plot(s, fr, "-o", ms=3, color=COLOR[r], label=SHORT[r])
    ax[1].axhline(b["noise_floor_rmse"], ls=":", lw=1.1, color=COLOR[r], alpha=0.8)
    ax[2].plot(s, fr / b["identity_rmse"], "-o", ms=3, color=COLOR[r], label=SHORT[r])
ax[1].set_xlabel("free-run step (0 = first unobserved frame)"); ax[1].set_ylabel("RMSE vs clean observation [0,1]")
ax[1].set_title("(a cell's own noise floor is its dotted line)\n(b) raw free-run RMSE — NOT comparable across cells",
                fontsize=10)
ax[1].legend(fontsize=7); ax[1].grid(alpha=0.3); style_ax(ax[1])
ax[2].axhline(1.0, color="0.3", ls="--", lw=1.2, label="that cell's copy-previous-frame baseline")
ax[2].set_xlabel("free-run step"); ax[2].set_ylabel("free-run RMSE ÷ own copy-previous-frame baseline")
ax[2].set_title("(c) the comparable view: each cell on its own scale", fontsize=10)
ax[2].legend(fontsize=7); ax[2].grid(alpha=0.3); style_ax(ax[2])
fig.suptitle("Fig 1 — predictive quality across the noise 2×2 (each cell read against its own dataset baselines)",
             y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig1_predictive.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

---
## §2 / §3 — Recoverability and canonicality across the 2×2

Two directional predictions to check: removing **sensing** noise should *raise* the fiber residual (with a clean view
there is no filtering pressure, so the state can afford idiosyncratic extra content), and removing **world** noise
should raise **velocity** R² most (velocity becomes exactly constant, hence perfectly inferable from history).

In [ ]:
# [3] Fig 2 + Table 1 — recoverability and canonicality, one bar per noise cell.
plt.style.use("default")
fig, ax = plt.subplots(1, 2, figsize=(14.5, 4.5))
gr1 = [("pos_r2_linear","position R²\n(linear)"), ("pos_r2_mlp","position R²\n(MLP)"),
       ("vel_r2_linear","velocity R²\n(linear)"), ("vel_r2_mlp","velocity R²\n(MLP)")]
gr2 = [("fiber_resid_linear","fiber residual\n(linear)"), ("fiber_resid_mlp","fiber residual\n(MLP)")]
for a, grp, ylab, ttl in [
        (ax[0], gr1, "R² (higher is better)", "(a) recoverability: is the physical state readable from h?"),
        (ax[1], gr2, "residual as a fraction of ‖h‖ (lower is better)",
         "(b) canonicality: how much of h is NOT a function of (position, velocity)?")]:
    xi = np.arange(len(grp)); w = 0.2
    for k, r in enumerate(RUNS):
        a.bar(xi + (k - 1.5)*w, [RES[r][q] for q, _ in grp], w, color=COLOR[r], label=SHORT[r])
    a.set_xticks(xi); a.set_xticklabels([lab for _, lab in grp], fontsize=8.5)
    a.set_ylabel(ylab); a.set_title(ttl, fontsize=10); a.axhline(0, color="0.3", lw=0.8)
    a.grid(alpha=0.3, axis="y"); style_ax(a)
ax[0].legend(fontsize=7.5, ncol=2)
fig.suptitle("Fig 2 — what the hidden state encodes under each combination of noise sources", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig2_recovery.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

rows = ["| cell | obs / pos noise | next-step RMSE (own noise floor) | position R² lin / MLP ↑ | velocity R² lin / MLP ↑ | fiber residual lin / MLP ↓ |",
        "|---|---|---|---|---|---|"]
for r in RUNS:
    d = RES[r]; b = d["baselines"]
    rows.append(f"| {SHORT[r]} | {d['obs_noise_std']:.2f} / {d['position_noise_std']:.2f} | "
                f"{d['nextstep_rmse_vs_clean']:.4f} ({b['noise_floor_rmse']:.4f}) | "
                f"{d['pos_r2_linear']:.3f} / {d['pos_r2_mlp']:.3f} | "
                f"{d['vel_r2_linear']:.3f} / {d['vel_r2_mlp']:.3f} | "
                f"{d['fiber_resid_linear']:.3f} / {d['fiber_resid_mlp']:.3f} |")
display(Markdown("**Table 1 — predictive quality, recoverability and canonicality across the noise 2×2.** "
                 "Next-step RMSE is shown with that cell's own observation noise floor in brackets; the two are "
                 "only meaningful together.\n\n" + "\n".join(rows)))

---
## §4 — Editability across the 2×2

The decisive axis is again **ghost ratio** (1.0 = the object never left). The **decoder-gradient oracle** is shown
beside the probe-directed editors as the bracket on each model.

In [ ]:
# [4] Fig 3 — editability across the noise 2×2: Edit Index headline, then the zone decomposition.
plt.style.use("default")
groups = ["Unsteered", "Oracle observation"] + EDITORS
panels = [("edit_index",      "Edit Index  (+1 edited world … −1 unedited world)",
           "(a) HEADLINE — did the edit land?"),
          ("target_rmse",     "Target RMSE vs ground truth",
           "(b) did the object appear at the target?"),
          ("ghost_rmse",      "Ghost RMSE vs ground truth",
           "(c) did it leave its old location?"),
          ("collateral_rmse", "Collateral RMSE vs ground truth",
           "(d) was the OTHER object left alone?"),
          ("gt_traj_rmse",    "GT-traj RMSE (mean over the rollout)",
           "(e) did the edit HOLD over the rollout?")]
xi = np.arange(len(groups)); w = 0.2
fig, ax = plt.subplots(1, 5, figsize=(27, 5.4))
for i, (key, ylab, ttl) in enumerate(panels):
    for k, r in enumerate(RUNS):
        ax[i].bar(xi + (k - 1.5)*w, [RES[r]["editability"][g][key] for g in groups], w,
                  color=COLOR[r], label=SHORT[r])
    ax[i].set_xticks(xi)
    ax[i].set_xticklabels([g + (" (ORACLE)" if g == ORACLE else
                           (" (reference)" if g in ("Unsteered", "Oracle observation") else "")) for g in groups],
                          fontsize=8, rotation=25, ha="right")
    ax[i].set_ylabel(ylab); ax[i].set_title(ttl, fontsize=10); ax[i].grid(alpha=0.3, axis="y"); style_ax(ax[i])
for y, lab in [(1.0, "edited world"), (0.0, "equidistant / garbage"), (-1.0, "unedited world")]:
    ax[0].axhline(y, color="0.4", ls=":", lw=1.0)
    ax[0].annotate(lab, xy=(len(groups)-0.5, y), fontsize=7, color="0.35", ha="right", va="bottom")
ax[0].set_ylim(-1.05, 1.05)
ax[0].legend(fontsize=7, ncol=2)
fig.suptitle("Fig 3 — editability under each combination of noise sources "
             "(probe-directed editors versus the decoder-gradient oracle bracket)", y=1.04, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3_editability.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

# Fig 3b — the Edit Index over the ROLLOUT: a step-0 win is not the same as an edit that holds.
fig, axes = plt.subplots(1, len(RUNS), figsize=(4.2*len(RUNS), 3.9), sharey=True)
sx = np.arange(K)
EC = {"Readout injection": "#0072B2", "Global-PCA projection": "#E69F00", "PCA geodesic": "#CC79A7",
      "MLP-probe gradient": "#56B4E9", "Decoder gradient": "#009E73"}
for a, r in zip(axes, RUNS):
    e = RES[r]["editability"]
    for ed in EDITORS:
        a.plot(sx, e[ed]["edit_index_by_step"], lw=2.4 if ed == ORACLE else 1.5, color=EC[ed],
               label=ed + (" (ORACLE)" if ed == ORACLE else ""))
    a.plot(sx, e["Unsteered"]["edit_index_by_step"], ":", color="0.55", lw=1.6, label="unsteered (no edit)")
    a.plot(sx, e["Oracle observation"]["edit_index_by_step"], "--", color="0.2", lw=1.6,
           label="oracle observation (reference)")
    a.axhline(0, color="0.4", ls=":", lw=1.0); a.set_ylim(-1.05, 1.05)
    a.set_title(SHORT[r], fontsize=9); a.set_xlabel("rollout step s (0 = frame ef)")
    a.grid(alpha=0.3); style_ax(a)
axes[0].set_ylabel("Edit Index")
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="upper center", ncol=7, fontsize=8, frameon=False, bbox_to_anchor=(0.5, 1.0))
fig.suptitle("Fig 3b — does the edit HOLD? Edit Index at every rollout step "
             "(+1 = the edited world, −1 = the unedited world, 0 = neither)", y=1.13, fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.90]); fig.savefig(f"{OUT}/fig3b_edit_index_rollout.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

In [ ]:
# [5] Fig 4 — per-step RMSE against the moving post-edit ground truth: one panel per editor, one line per cell.
plt.style.use("default")
panels = ["Unsteered", "Oracle observation"] + EDITORS
fig, axes = plt.subplots(2, 4, figsize=(18, 7.6), sharex=True, sharey=True)
s = np.arange(K)
for a, ed in zip(axes.ravel(), panels):
    for r in RUNS:
        a.plot(s, RES[r]["editability"][ed]["step_rmse_to_gt"], color=COLOR[r], lw=1.6, label=SHORT[r])
    ttl = ed + (" (ORACLE — the bracket)" if ed == ORACLE else
                (" (reference)" if ed in ("Unsteered", "Oracle observation") else ""))
    a.set_title(ttl, fontsize=9.5); a.grid(alpha=0.3); style_ax(a)
for a in axes[-1]: a.set_xlabel("rollout step s (0 = sim frame ef)")
for a in axes[:, 0]: a.set_ylabel("RMSE vs clean_obs[ef+s]")
axes.ravel()[-1].axis("off")
h, l = axes[0][0].get_legend_handles_labels()
fig.legend(h, l, loc="lower right", ncol=1, fontsize=9.5, frameon=False, bbox_to_anchor=(0.95, 0.12))
fig.suptitle("Fig 4 — does the edit land and hold? RMSE against the time-evolving true post-edit observation",
             y=1.0, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig4_steprmse.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

In [ ]:
# [6] Canonical observation-waterfall helper (CLAUDE.md fixed spec): gray on dark, 6 noisy context frames above a
#     dashed edit line, the shared TRUE post-edit row, then each column's OWN free-run; green target / red-dash ghost.
N_CTX = 6
DARK, TXT, TICK, EDIT_C, ROW_C = "#0a0a14", "#a3adc2", "#808a9d", "#fa8850", "#FFD166"
TARGET_C, GHOST_C = "#00E676", "#FF5252"

def waterfall_grid(col_titles, rows, suptitle, fname):
    """rows[i] = dict(label, ctx (N_CTX,R), bodies list[(K,R)], tgt_cx, ghost_cx).

    Each column shows its OWN free-run from step 0 (which decodes sim frame `ef`) — there is
    deliberately NO shared teacher-forced `ef` row: only the Oracle observation reference ever
    sees that frame, and painting it into every column would hide the exact frame §4 scores.
    Each row is self-contained (own context frames, locators and GT column), which is what lets
    rows come from different models/datasets as well as from different samples."""
    ncol = len(col_titles)
    fig, axes = plt.subplots(len(rows), ncol, figsize=(2.9*ncol, 3.3*len(rows)),
                             squeeze=False, facecolor=DARK)
    for r, row in enumerate(rows):
        for c in range(ncol):
            ax = axes[r][c]; ax.set_facecolor(DARK)
            panel = np.clip(np.concatenate([row["ctx"], row["bodies"][c]], 0), 0, 1)
            ax.imshow(panel, aspect="auto", origin="upper", cmap="gray", vmin=0, vmax=1, interpolation="nearest")
            for sp in ax.spines.values(): sp.set_edgecolor(TICK)
            ax.axhline(N_CTX-0.5, color=EDIT_C, lw=1.4, ls="--", alpha=0.95)
            if not np.isnan(row["tgt_cx"]):   ax.axvline(row["tgt_cx"],   color=TARGET_C, lw=1.6, alpha=0.9)
            if not np.isnan(row["ghost_cx"]): ax.axvline(row["ghost_cx"], color=GHOST_C, ls="--", lw=1.6, alpha=0.9)
            if r == 0: ax.set_title(col_titles[c], fontsize=8, color=TXT)
            if c == 0:
                ax.set_ylabel(row["label"], fontsize=7.5, color=TXT)
                ax.set_yticks([0, N_CTX, N_CTX+7, N_CTX+14])
                ax.set_yticklabels([ef-N_CTX, ef, ef+7, ef+14], fontsize=7)
            else: ax.set_yticks([])
            ax.set_xlabel("ray", fontsize=8, color=TXT); ax.tick_params(colors=TICK, labelsize=7)
    handles = [Line2D([0],[0], color=TARGET_C, lw=2.2, label="object target location"),
               Line2D([0],[0], color=GHOST_C, ls="--", lw=2.2, label="ghost (pre-edit) location"),
               Line2D([0],[0], color=EDIT_C, ls="--", lw=2.2,
                      label=f"edit applied here ({N_CTX} noisy context frames above; every row below is that "
                            f"column's OWN free-run, step 0 = frame {ef})")]
    fig.legend(handles=handles, loc="upper center", ncol=2, fontsize=8.5, frameon=False,
               labelcolor=TXT, bbox_to_anchor=(0.5, 0.955))
    fig.suptitle(suptitle, y=1.0, fontsize=10.5, color=TXT)
    fig.tight_layout(rect=[0, 0, 1, 0.90])
    fig.savefig(f"{OUT}/{fname}", dpi=130, bbox_inches="tight", facecolor=DARK)
    display(fig); plt.close(fig); print("saved", fname)
print("waterfall helper ready")

In [ ]:
# [7] Fig 5 — waterfalls: one ROW per noise cell (each with its own GT column, since the cells are different
#     datasets), columns = the editor line-up. Sample chosen per cell as its largest teleport with a visible ghost.
cols = ["GT (sim)", "unsteered\n(no edit)", "oracle observation\n(reference)",
        "PCA geodesic", "MLP-probe\ngradient", "Decoder gradient\n(ORACLE)"]
keys = [None, "Unsteered", "Oracle observation", "PCA geodesic", "MLP-probe gradient", "Decoder gradient"]
rows = []
for r in RUNS:
    z = NPZ[r]
    smp = int(np.argsort(z["teleport"] * (z["n_ghost_rays"] >= 3))[::-1][0])
    rows.append(dict(
        label=f"{SHORT[r]}\nsample {smp} (teleport {z['teleport'][smp]:.1f})\nsim frame",
        ctx=z["ctx"][smp],
        tgt_cx=z["tgt_cx"][smp], ghost_cx=z["ghost_cx"][smp],
        bodies=[z["gt_roll"][smp] if k is None else z[f"roll_{k}"][smp] for k in keys]))
waterfall_grid(cols, rows,
               "Fig 5 — post-edit observation rollouts, one row per noise cell "
               "(each row is a different dataset, so each has its own ground-truth column)",
               "fig5_waterfalls.png")

---
## §5 — Summary

Four questions, answered from the numbers below: does removing sensing noise cost canonicality; does removing world
noise buy velocity readability; is the editability negative sensitive to either; and does the oracle still bracket the
structural editors in every cell.

In [ ]:
# [8] Table 2 + data-driven verdict, on the canonical §4 set (Edit Index + zone RMSEs + fidelity ratio).
rows = ["| cell | editor | Edit Index ↑ | Target RMSE ↓ | Ghost RMSE ↓ | Collateral RMSE ↓ | GT-traj RMSE ↓ | fidelity ratio |",
        "|---|---|---|---|---|---|---|---|"]
BEST, VALID = {}, {}
for r in RUNS:
    e = RES[r]["editability"]
    # a structural editor only counts as having made an edit if it did not DEGRADE the rollout
    # (fidelity ratio <= 1); otherwise a move toward index 0 is destruction, not relocation.
    ok = [ed for ed in STRUCTURAL if e[ed]["fidelity_ratio"] <= 1.0]
    BEST[r] = max(ok or STRUCTURAL, key=lambda ed: e[ed]["edit_index"])
    VALID[r] = bool(ok)
    for ed in ["Unsteered", "Oracle observation"] + STRUCTURAL + [ORACLE]:
        c = e[ed]
        note = " *(destroys the observation)*" if max(c["target_rmse"], c["ghost_rmse"]) > 1.0 else ""
        rows.append(f"| {SHORT[r]} | {ed}{note} | **{c['edit_index']:+.2f}** | {c['target_rmse']:.3f} | "
                    f"{c['ghost_rmse']:.3f} | {c['collateral_rmse']:.3f} | {c['gt_traj_rmse']:.3f} | "
                    f"{c['fidelity_ratio']:.2f} |")
display(Markdown("**Table 2 — editability across the noise 2×2.** Edit Index: +1 = the output is the world where the "
                 "edit happened, −1 = the world where it did not, 0 = equidistant from both (ambiguous or garbage). "
                 "Observation intensity is bounded in [0,1], so a zone RMSE above 1 means the edit pushed the scan out "
                 "of range entirely.\n\n" + "\n".join(rows)))

print("================ VERDICT (computed, not asserted) ================")
print("1. SENSING NOISE AND CANONICALITY -- the two estimators disagree in sign, so both are reported:")
for r in RUNS:
    print(f"     {SHORT[r]:<36s} fiber residual linear {RES[r]['fiber_resid_linear']:.3f} | "
          f"MLP {RES[r]['fiber_resid_mlp']:.3f}")
dl = RES["N_obs0_pos004"]["fiber_resid_linear"] - RES["H256"]["fiber_resid_linear"]
dm = RES["N_obs0_pos004"]["fiber_resid_mlp"] - RES["H256"]["fiber_resid_mlp"]
print(f"   Turning sensing noise OFF (world noise held on) moves the linear residual by {dl:+.3f} and the MLP residual "
      f"by {dm:+.3f}. Under the MLP estimator -- the one that can follow a curved embedding -- sensing noise makes the "
      f"state markedly MORE canonical; the linear estimator shows a much smaller move in the opposite direction. "
      f"Read as: sensing noise reorganises the code into something an MLP can invert, not into a linear one.")

dv = RES["N_obs02_pos0"]["vel_r2_linear"] - RES["H256"]["vel_r2_linear"]
print(f"2. WORLD NOISE AND VELOCITY: linear velocity R² {RES['H256']['vel_r2_linear']:.3f} with world noise vs "
      f"{RES['N_obs02_pos0']['vel_r2_linear']:.3f} without ({dv:+.3f}) -- removing world noise does NOT buy velocity "
      f"readability, contrary to the pre-registered expectation that a constant-velocity world would be easier to read.")
print(f"   The larger recoverability effect is from SENSING noise on POSITION: linear position R² "
      f"{RES['N_obs0_pos0']['pos_r2_linear']:.3f} (no noise) vs {RES['N_obs02_pos0']['pos_r2_linear']:.3f} "
      f"(sensing noise only) -- observation noise acts as a regulariser that pushes position into a linearly "
      f"readable code.")

print("3. EDITABILITY -- Edit Index, best probe-directed editor vs the oracle, by cell:")
for r in RUNS:
    e = RES[r]["editability"]
    b = BEST[r]; cb, cu, co = e[b], e["Unsteered"], e[ORACLE]
    wrecked = max(cb["target_rmse"], cb["ghost_rmse"]) > 1.0
    print(f"     {SHORT[r]:<36s} unsteered {cu['edit_index']:+.2f} | best structural {cb['edit_index']:+.2f} ({b})"
          f"{'  <- NONE pass the fidelity guard; this one degrades the rollout' if wrecked else ''} | oracle {co['edit_index']:+.2f}")
gap = [RES[r]["editability"][BEST[r]]["edit_index"] - RES[r]["editability"]["Unsteered"]["edit_index"] for r in RUNS]
oi = [RES[r]["editability"][ORACLE]["edit_index"] for r in RUNS]
print("   => " + ("NEITHER noise source is what blocks editing: in every cell -- including the fully deterministic, "
                  "perfectly-sensed world -- the probe-directed editors stay near the unsteered end of the index while "
                  "the oracle reaches the edited world on the same model and decoder."
                  if max(gap) < 0.6 and min(oi) > 0.5 else
                  "a noise cell DOES move grabbability -- inspect Table 2, Fig 3a and Fig 5 before concluding."))
print("4. BELIEF INERTIA IS A FUNCTION OF SENSING NOISE -- the oracle observation (one frame of genuine teleport "
      "evidence) updates the belief much further when observations are clean:")
for r in RUNS:
    print(f"     {SHORT[r]:<36s} swap Edit Index {RES[r]['editability']['Oracle observation']['edit_index']:+.2f}")
print("\nPNGs:", sorted(os.listdir(OUT)))